In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2023_Mandir_Marg_Delhi_DPCC_2023.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,254.0,161.0,NaN,186.0,157.0,80.0,61.0,74.0,126.0,141.0,362.0,365.0
1,2,NaN,177.0,167.0,94.0,121.0,95.0,56.0,89.0,125.0,129.0,358.0,346.0
2,3,390.0,184.0,122.0,192.0,98.0,98.0,NaN,73.0,134.0,128.0,NaN,318.0
3,4,359.0,231.0,NaN,121.0,88.0,199.0,127.0,88.0,135.0,148.0,416.0,300.0
4,5,353.0,240.0,NaN,164.0,154.0,161.0,88.0,73.0,122.0,148.0,445.0,279.0
5,6,406.0,203.0,126.0,126.0,208.0,122.0,62.0,84.0,105.0,173.0,NaN,277.0
6,7,387.0,258.0,166.0,134.0,126.0,240.0,63.0,95.0,64.0,161.0,372.0,286.0
7,8,378.0,NaN,199.0,130.0,NaN,NaN,62.0,110.0,70.0,125.0,434.0,315.0
8,9,449.0,192.0,228.0,168.0,176.0,140.0,58.0,110.0,35.0,NaN,424.0,302.0
9,10,424.0,179.0,178.0,150.0,186.0,120.0,NaN,128.0,31.0,NaN,286.0,302.0


In [4]:
df.shape
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Day        40 non-null     object 
 1   January    35 non-null     float64
 2   February   31 non-null     float64
 3   March      33 non-null     float64
 4   April      34 non-null     float64
 5   May        35 non-null     float64
 6   June       34 non-null     float64
 7   July       24 non-null     float64
 8   August     34 non-null     float64
 9   September  34 non-null     float64
 10  October    35 non-null     float64
 11  November   32 non-null     float64
 12  December   35 non-null     float64
dtypes: float64(12), object(1)
memory usage: 4.3+ KB


In [5]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [6]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [7]:
# Define a function for outlier handling
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            # Replace outliers with mean
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [8]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready.head()

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,254.0,161.0,144.424242,186.000000,157.0,80.000000,61.000000,74.0,126.0,141.0,362.00000,365.0
1,2,254.6,177.0,167.000000,149.470588,121.0,95.000000,56.000000,89.0,125.0,129.0,358.00000,346.0
2,3,390.0,184.0,122.000000,192.000000,98.0,98.000000,59.791667,73.0,134.0,128.0,307.21875,318.0
3,4,359.0,231.0,144.424242,121.000000,88.0,108.323529,59.791667,88.0,135.0,148.0,416.00000,300.0
4,5,353.0,240.0,144.424242,164.000000,154.0,161.000000,59.791667,73.0,122.0,148.0,445.00000,279.0


In [9]:
df_ml_ready.shape
df_ml_ready.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Day        31 non-null     int64  
 1   January    31 non-null     float64
 2   February   31 non-null     float64
 3   March      31 non-null     float64
 4   April      31 non-null     float64
 5   May        31 non-null     float64
 6   June       31 non-null     float64
 7   July       31 non-null     float64
 8   August     31 non-null     float64
 9   September  31 non-null     float64
 10  October    31 non-null     float64
 11  November   31 non-null     float64
 12  December   31 non-null     float64
dtypes: float64(12), int64(1)
memory usage: 3.3 KB
